In [88]:
import pandas as pd
import seaborn as sns

In [89]:
titanic_raw = sns.load_dataset("titanic")
titanic_raw.to_csv("data/raw/titanic.csv",index=False)
print("저장 완료")

저장 완료


In [90]:
titanic = pd.read_csv("data/raw/titanic.csv")
print(titanic.shape)
print(titanic.head(3))

(891, 15)
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  


## 1.열 선택

In [91]:
# 1.
print(type(titanic["fare"]))    # 1차원 배열 series
print(titanic["fare"].dtype)
# 2.
df= titanic[["survived","pclass","sex","age"]] # 리스트 다중 열
print(type(df))        # 2차원 데이터 - dataframe
print(df.dtypes)
print("shape",df)

#

<class 'pandas.Series'>
float64
<class 'pandas.DataFrame'>
survived      int64
pclass        int64
sex             str
age         float64
dtype: object
shape      survived  pclass     sex   age
0           0       3    male  22.0
1           1       1  female  38.0
2           1       3  female  26.0
3           1       1  female  35.0
4           0       3    male  35.0
..        ...     ...     ...   ...
886         0       2    male  27.0
887         1       1  female  19.0
888         0       3  female   NaN
889         1       1    male  26.0
890         0       3    male  32.0

[891 rows x 4 columns]


## 2. loc vs iloc

In [92]:
# 1.
df = titanic.loc[10:15,["sex","age"]] #loc 이름 기준 -표기자리 이하
print(df)
# 2.
idf= titanic.iloc[10:15,[2,3]] # iloc 파이썬 인덱스 방식 - 표기자리 미만
print(idf)

       sex   age
10  female   4.0
11  female  58.0
12    male  20.0
13    male  39.0
14  female  14.0
15  female  55.0
       sex   age
10  female   4.0
11  female  58.0
12    male  20.0
13    male  39.0
14  female  14.0


## 3. iloc 응용

In [93]:
idf= titanic.iloc[-3:,4]
print(idf)

888    1
889    0
890    0
Name: sibsp, dtype: int64


## 4.조건 마스크

In [94]:
# 1.
mask = titanic["age"] < 10
m = mask.head(5)
print(m)
# 2.
t = len(titanic[mask])
# t=mask.sum()
print("인원수: ",t)

0    False
1    False
2    False
3    False
4    False
Name: age, dtype: bool
인원수:  62


## 5.복합조건

In [95]:
# 1.
survive = (titanic["pclass"] == 3) & (titanic["sex"] == "male") & (titanic["survived"] == 1)
print(sum(survive))
print(survive.sum())
# 2.
cf = (titanic["pclass"] == 1) | (titanic["fare"] >= 100 )
print(sum(cf))
print(cf.sum())
# 3.
al = titanic["alone"] ==True
print(sum(~al))
print((~al).sum())

47
47
216
216
354
354


## 6.isin.between

In [96]:
# 1. 
town = titanic["embark_town"].isin(["Cherbourg","Queenstown"])
print(titanic[town].shape)
# 2.
age = titanic["age"].between(20 , 30)
print(titanic[age].shape)
print(titanic[age]["age"].min())
print(titanic[age]["age"].max())
# 3.
nage=age = titanic["age"].between(20 , 30, inclusive = "neither")
print(titanic[age].shape)

(245, 15)
(245, 15)
20.0
30.0
(205, 15)


## 7. loc[조건.,열]

In [97]:
mask = (titanic["age"] <= 12) & (titanic["pclass"] == 3)
kids = titanic.loc[mask, ["age", "sex", "survived"]]
print("행:", len(kids))
survive = round(kids["survived"].mean(),2)
print("생존율:",survive)

행: 48
생존율: 0.42


## 8.query

In [98]:
# 1.
min_fare =50
df = titanic.query("embark_town == 'Cherbourg' and fare >= @min_fare ")
print(df.shape)
# 2.
survive = titanic.query("pclass == 3 and sex == 'male' and survived == 1")

print(len(survive)) # 47명 출력으로 같음

(61, 15)
47


## 9.정렬

In [99]:
# 1.
df= titanic.sort_values(by=["pclass", "fare"], ascending=[True, False]
)
result = df[["pclass", "fare", "sex", "age"]].head(3)
print(result)

# 2.
sort = titanic.sort_values(by="fare", ascending=False)
iloc_fare = sort.iloc[0]["fare"]
loc_fare = sort.loc[0]["fare"]
print("iloc:", iloc_fare)   #iloc  데이터표 순서 기준 - 1등 요금
print("loc :", loc_fare)    #loc - 인덱스 번호 기준


     pclass      fare     sex   age
258       1  512.3292  female  35.0
679       1  512.3292    male  36.0
737       1  512.3292    male  35.0
iloc: 512.3292
loc : 7.25


## 10.종합 __ 20~40세 남성, 1.2등석 vs 3등석

In [100]:
# 1. 
mask =(
    (titanic["sex"] == "male")
    & (titanic["age"].between(20, 40))
    & (titanic["pclass"].isin([1, 2]))
)
df= titanic.loc[mask, ["pclass", "age", "fare", "survived"]]
# 2.
print("행 수 :", len(df))
print("1,2등석 생존률 :", round(df["survived"].mean(), 2))
print("요금 상위 3명 :",df.sort_values(by="fare", ascending=False).head(3))


# 3.
query = titanic.query(
    "sex == 'male' and age.between(20, 40) and pclass.isin([1, 2])"
)
print("query 행 수 :", len(query))
print("query:",len(query)) # 일치
print("df:",len(df))


# 4.

mask = (
    (titanic["sex"] == "male")
    & (titanic["age"].between(20, 40))
    & (titanic["pclass"] == 3)
)
ndf = titanic.loc[mask, ["pclass", "age", "fare", "survived"]]


print("3등석 생존률 :", round(ndf["survived"].mean(), 2))


행 수 : 103
생존률 : 0.24
요금 상위 3명 :      pclass   age      fare  survived
679       1  36.0  512.3292         1
737       1  35.0  512.3292         1
118       1  24.0  247.5208         0
query 행 수 : 103
query: 103
df: 103
생존률 : 0.15


20~40세 젊은 남성 승객의 생존율을 분석 결과, 1·2등석 승객의 생존율은 약 0.24 3등석 승객의 생존율은 약 0.15 로 확인 
이를 토대로 객실의 등급 차이가 생존율에 영향을 미치는 것으로 확인

사회적인 격차가 생존율에 영향을 미쳣다고 해석가능